# LLM Evaluation with MLflow Example Notebook

In this notebook, we will demonstrate how to evaluate various LLMs and RAG systems with MLflow, leveraging simple metrics such as toxicity, as well as LLM-judged metrics such as relevance, and even custom LLM-judged metrics such as professionalism

We need to set our OpenAI API key, since we will be using GPT-4 for our LLM-judged metrics.

In order to set your private key safely, please be sure to either export your key through a command-line terminal for your current instance, or, for a permanent addition to all user-based sessions, configure your favored environment management configuration file (i.e., .bashrc, .zshrc) to have the following entry:

`OPENAI_API_KEY=<your openai API key>`

In [31]:
%reload_ext autoreload
%autoreload 2

import getpass
import os
import sys
from pathlib import Path

import mlflow
import openai
import pandas as pd
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage


In [32]:
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
#os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI API key (for evaluation):")
os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter MISTRAL API key (for model):")

In [33]:
# Set up MLflow experiment
mlflow.set_experiment("6-question-answering-evaluation")

<Experiment: artifact_location='mlflow-artifacts:/749975181582978121', creation_time=1750241844441, experiment_id='749975181582978121', last_update_time=1750241844441, lifecycle_stage='active', name='6-question-answering-evaluation', tags={}>

## Basic Question-Answering Evaluation

Create a test case of `inputs` that will be passed into the model and `ground_truth` which will be used to compare against the generated output from the model.

In [34]:
eval_df = pd.DataFrame(
    {
        "inputs": [
            "How does useEffect() work?",
            "What does the static keyword in a function mean?",
            "What does the 'finally' block in Python do?",
            "What is the difference between multiprocessing and multithreading?",
        ],
        "ground_truth": [
            "The useEffect() hook tells React that your component needs to do something after render. React will remember the function you passed (we’ll refer to it as our “effect”), and call it later after performing the DOM updates.",
            "Static members belongs to the class, rather than a specific instance. This means that only one instance of a static member exists, even if you create multiple objects of the class, or if you don't create any. It will be shared by all objects.",
            "'Finally' defines a block of code to run when the try... except...else block is final. The finally block will be executed no matter if the try block raises an error or not.",
            "Multithreading refers to the ability of a processor to execute multiple threads concurrently, where each thread runs a process. Whereas multiprocessing refers to the ability of a system to run multiple processors in parallel, where each processor can run one or more threads.",
        ],
    }
)

Create a Mistral-based model that asks Mistral to answer the question in two sentences. Call `mlflow.evaluate()` with the model and evaluation dataframe.

In [35]:
with mlflow.start_run() as run:
    system_prompt = "Answer the following question in two sentences"

    # Remove code_path argument
    basic_qa_model = mlflow.pyfunc.log_model(
        python_model="../src/mistral_wrapper.py",
        artifact_path="model",
        input_example={"question": ["What is MLflow?"]}
    )

    results = mlflow.evaluate(
        model=basic_qa_model.model_uri,
        data=eval_df,
        targets="ground_truth",
        model_type="question-answering",
        feature_names=["inputs"],
        evaluators="default",
    )
results.metrics

2025/06/19 00:18:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025/06/19 00:18:03 INFO mlflow.pyfunc: Inferring model signature from input example
2025/06/19 00:18:03 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: KeyError('inputs'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.
2025/06/19 00:18:05 WARNING mlflow.utils.requirements_utils: Failed to run predict on input_example, dependencies introduced in predict are

🏃 View run masked-gull-436 at: http://127.0.0.1:5000/#/experiments/749975181582978121/runs/5cd5ec5487704b05b0ec5ea7f5e0743d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/749975181582978121


{'exact_match/v1': 0.0}

Inspect the evaluation results table as a dataframe to see row-by-row metrics to further assess model performance

In [36]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count
0,How does useEffect() work?,The useEffect() hook tells React that your com...,UseEffect() is a built-in hook in React that l...,75
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...",The `static` keyword in a function means that ...,64
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The 'finally' block in Python is used to ensur...,45
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,Multiprocessing involves creating separate pro...,61


## LLM-judged correctness with OpenAI GPT-4

Construct an answer similarity metric using the `answer_similarity()` metric factory function.

In [37]:
from mlflow.metrics.genai import EvaluationExample, answer_similarity

# Create an example to describe what answer_similarity means like for this problem.
example = EvaluationExample(
    input="What is MLflow?",
    output="MLflow is an open-source platform for managing machine "
    "learning workflows, including experiment tracking, model packaging, "
    "versioning, and deployment, simplifying the ML lifecycle.",
    score=4,
    justification="The definition effectively explains what MLflow is "
    "its purpose, and its developer. It could be more concise for a 5-score.",
    grading_context={
        "targets": "MLflow is an open-source platform for managing "
        "the end-to-end machine learning (ML) lifecycle. It was developed by Databricks, "
        "a company that specializes in big data and machine learning solutions. MLflow is "
        "designed to address the challenges that data scientists and machine learning "
        "engineers face when developing, training, and deploying machine learning models."
    },
)

# Construct the metric using OpenAI GPT-4 as the judge
answer_similarity_metric = answer_similarity(model="mistral:/mistral-tiny", examples=[example])

print(answer_similarity_metric)

EvaluationMetric(name=answer_similarity, greater_is_better=True, long_name=answer_similarity, version=v1, metric_details=
Task:
You must return the following fields in your response in two lines, one below the other:
score: Your numerical score for the model's answer_similarity based on the rubric
justification: Your reasoning about the model's answer_similarity score

You are an impartial judge. You will be given an input that was sent to a machine
learning model, and you will be given an output that the model produced. You
may also be given additional information that was used by the model to generate the output.

Your task is to determine a numerical score called answer_similarity based on the input and output.
A definition of answer_similarity and a grading rubric are provided below.
You must use the grading rubric to determine your score. You must also justify your score.

Examples could be included below for reference. Make sure to use them as references and to
understand them be

Call `mlflow.evaluate()` again but with your new `answer_similarity_metric`

In [38]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        basic_qa_model.model_uri,
        eval_df,
        targets="ground_truth",
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[answer_similarity_metric],  # use the answer similarity metric created above
    )
results.metrics

C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025/06/19 00:18:10 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-a9b4409795254d3a929e6b186b98610a
2025/06/19 00:18:10 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/06/19 00:18:10 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/06/19 00:18:14 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/06/19 00:18:14 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: ModuleNotFoundError

🏃 View run capable-kite-60 at: http://127.0.0.1:5000/#/experiments/749975181582978121/runs/d81269b7be654cbfa6417ab939e77972
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/749975181582978121


{'exact_match/v1': 0.0,
 'answer_similarity/v1/mean': np.float64(3.5),
 'answer_similarity/v1/variance': np.float64(0.25),
 'answer_similarity/v1/p90': np.float64(4.0)}

See the row-by-row LLM-judged answer similarity score and justifications

In [39]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,answer_similarity/v1/score,answer_similarity/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,UseEffect() is a built-in hook in React that l...,82,3,The output provides a general description of U...
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...","In programming, the ""static"" keyword in a func...",57,3,The model's output correctly explains the conc...
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The 'finally' block in Python is executed afte...,50,4,The output provides a clear and accurate descr...
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,Multiprocessing in Python involves creating se...,65,4,The output provides a clear explanation of the...


## Custom LLM-judged metric for professionalism

Create a custom metric that will be used to determine professionalism of the model outputs. Use `make_genai_metric` with a metric definition, grading prompt, grading example, and judge model configuration

In [40]:
from mlflow.metrics.genai import EvaluationExample, make_genai_metric

professionalism_metric = make_genai_metric(
    name="professionalism",
    definition=(
        "Professionalism refers to the use of a formal, respectful, and appropriate style of communication that is tailored to the context and audience. It often involves avoiding overly casual language, slang, or colloquialisms, and instead using clear, concise, and respectful language"
    ),
    grading_prompt=(
        "Professionalism: If the answer is written using a professional tone, below "
        "are the details for different scores: "
        "- Score 1: Language is extremely casual, informal, and may include slang or colloquialisms. Not suitable for professional contexts."
        "- Score 2: Language is casual but generally respectful and avoids strong informality or slang. Acceptable in some informal professional settings."
        "- Score 3: Language is balanced and avoids extreme informality or formality. Suitable for most professional contexts. "
        "- Score 4: Language is noticeably formal, respectful, and avoids casual elements. Appropriate for business or academic settings. "
        "- Score 5: Language is excessively formal, respectful, and avoids casual elements. Appropriate for the most formal settings such as textbooks. "
    ),
    examples=[
        EvaluationExample(
            input="What is MLflow?",
            output=(
                "MLflow is like your friendly neighborhood toolkit for managing your machine learning projects. It helps you track experiments, package your code and models, and collaborate with your team, making the whole ML workflow smoother. It's like your Swiss Army knife for machine learning!"
            ),
            score=2,
            justification=(
                "The response is written in a casual tone. It uses contractions, filler words such as 'like', and exclamation points, which make it sound less professional. "
            ),
        )
    ],
    version="v1",
    model="mistral:/mistral-tiny",
    parameters={"temperature": 0.0},
    grading_context_columns=[],
    aggregations=["mean", "variance", "p90"],
    greater_is_better=True,
)

print(professionalism_metric)

EvaluationMetric(name=professionalism, greater_is_better=True, long_name=professionalism, version=v1, metric_details=
Task:
You must return the following fields in your response in two lines, one below the other:
score: Your numerical score for the model's professionalism based on the rubric
justification: Your reasoning about the model's professionalism score

You are an impartial judge. You will be given an input that was sent to a machine
learning model, and you will be given an output that the model produced. You
may also be given additional information that was used by the model to generate the output.

Your task is to determine a numerical score called professionalism based on the input and output.
A definition of professionalism and a grading rubric are provided below.
You must use the grading rubric to determine your score. You must also justify your score.

Examples could be included below for reference. Make sure to use them as references and to
understand them before complet

Call `mlflow.evaluate` with your new professionalism metric. 

In [41]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        basic_qa_model.model_uri,
        eval_df,
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[professionalism_metric],  # use the professionalism metric we created above
    )
print(results.metrics)

C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025/06/19 00:18:18 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-a9b4409795254d3a929e6b186b98610a
2025/06/19 00:18:18 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/06/19 00:18:18 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/06/19 00:18:21 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/06/19 00:18:21 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: ModuleNotFoundError

🏃 View run gentle-lamb-349 at: http://127.0.0.1:5000/#/experiments/749975181582978121/runs/f8b68d6993c2435fa1a4327386a48147
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/749975181582978121
{'professionalism/v1/mean': np.float64(3.0), 'professionalism/v1/variance': np.float64(0.0), 'professionalism/v1/p90': np.float64(3.0)}


In [42]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,professionalism/v1/score,professionalism/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,useEffect() is a built-in hook in React that a...,56,3,"The response is written in a balanced tone, av..."
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...","The ""static"" keyword in a function (particular...",67,3,"The response is written in a balanced tone, av..."
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The 'finally' block in Python is executed afte...,53,3,"The response is written in a balanced tone, av..."
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,Multiprocessing involves running separate proc...,66,3,"The response is written in a balanced tone, av..."


Lets see if we can improve `basic_qa_model` by creating a new model that could perform better by changing the system prompt.

Call `mlflow.evaluate()` using the new model. Observe that the professionalism score has increased!

In [43]:

with mlflow.start_run() as run:
    system_prompt = "Answer the following question in two sentences"

    basic_qa_model = mlflow.pyfunc.log_model(
        python_model="../src/mistral_wrapper.py",
        artifact_path="model",
        input_example={"question": ["What is MLflow?"]}
    )

    results = mlflow.evaluate(
        model=basic_qa_model.model_uri,
        data=eval_df,
        targets="ground_truth",
        model_type="question-answering",
        feature_names=["inputs"],
        evaluators="default",
        extra_metrics=[professionalism_metric],
    )
results.metrics

2025/06/19 00:18:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025/06/19 00:18:24 INFO mlflow.pyfunc: Inferring model signature from input example
2025/06/19 00:18:24 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: KeyError('inputs'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.
2025/06/19 00:18:26 WARNING mlflow.utils.requirements_utils: Failed to run predict on input_example, dependencies introduced in predict are

🏃 View run industrious-cow-903 at: http://127.0.0.1:5000/#/experiments/749975181582978121/runs/f628120fd8fa45498de66242f21d4e41
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/749975181582978121


{'exact_match/v1': 0.0,
 'professionalism/v1/mean': np.float64(3.0),
 'professionalism/v1/variance': np.float64(0.0),
 'professionalism/v1/p90': np.float64(3.0)}

In [44]:
results.tables["eval_results_table"]


,inputs,ground_truth,outputs,token_count,professionalism/v1/score,professionalism/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,`useEffect()` in React is a built-in hook that...,71,3,"The response is written in a balanced tone, av..."
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...","The ""static"" keyword in a function means that ...",48,3,"The response is written in a balanced tone, av..."
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The 'finally' block in Python is executed afte...,53,3,"The response is written in a balanced tone, av..."
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,Multiprocessing in Python involves creating se...,68,3,"The response is written in a balanced tone, av..."
